In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso, LassoCV
from sklearn.model_selection import RepeatedKFold
from sklearn.metrics import mean_squared_error

# -----------------------------
# 1. User-defined settings
# -----------------------------
input_file = "processed_data/train_imputed.csv"
output_dir = "feature_selection_results"
outcome_var = "outcome"
random_state = 42

# Study variables used in the current analysis
candidate_features = [
    "Age",
    "Gender",
    "Educational",
    "Occupation",
    "Height",
    "Weight",
    "BMI",
    "Waist_circumference",
    "Hip_circumference",
    "Hypertension",
    "DM",
    "HBV",
    "HCV"
]

# Final retained variables reported in the manuscript
final_retained_features = [
    "Age",
    "Gender",
    "Educational",
    "Height",
    "Weight",
    "BMI",
    "Waist_circumference",
    "Hip_circumference",
    "Hypertension",
    "DM"
]

# Example categorical variables
categorical_vars = [
    "Gender",
    "Educational",
    "Occupation",
    "Hypertension",
    "DM",
    "HBV",
    "HCV",
    outcome_var
]

# Alpha search space
alphas = np.logspace(-4, 0, 50)

# -----------------------------
# 2. Create output directory
# -----------------------------
os.makedirs(output_dir, exist_ok=True)

# -----------------------------
# 3. Read data
# -----------------------------
df = pd.read_csv(input_file)

print("Input data shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

# -----------------------------
# 4. Basic checks
# -----------------------------
if outcome_var not in df.columns:
    raise ValueError(f"Outcome variable '{outcome_var}' not found in dataset.")

available_features = [col for col in candidate_features if col in df.columns]
missing_features = [col for col in candidate_features if col not in df.columns]

if missing_features:
    print("Warning: the following candidate features were not found and will be ignored:")
    print(missing_features)

if len(available_features) == 0:
    raise ValueError("No candidate features found in the input dataset.")

# Keep only outcome + available candidate features
df = df[[outcome_var] + available_features].copy()

# -----------------------------
# 5. Convert categorical variables
# -----------------------------
available_categorical = [col for col in categorical_vars if col in df.columns]

for col in available_categorical:
    df[col] = df[col].astype("category")

# One-hot encode categorical predictors
X = df.drop(columns=[outcome_var])
X = pd.get_dummies(X, drop_first=True)

# Outcome variable
y = df[outcome_var]

# If outcome is categorical/binary, convert to numeric codes
if str(y.dtype) == "category" or y.dtype == object:
    y = pd.Categorical(y).codes

print("\nEncoded feature matrix shape:", X.shape)

# Save encoded feature names
pd.DataFrame({"Encoded_Features": X.columns}).to_csv(
    os.path.join(output_dir, "lasso_encoded_feature_names.csv"),
    index=False
)

# -----------------------------
# 6. Standardize predictors
# -----------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# -----------------------------
# 7. Initial LASSO model
# -----------------------------
lasso = Lasso(alpha=0.1, max_iter=10000, random_state=random_state)
lasso.fit(X_scaled, y)

coefficients = pd.Series(lasso.coef_, index=X.columns)
selected_features_initial = coefficients[coefficients != 0].index.tolist()

print("\nSelected features from initial LASSO:")
print(selected_features_initial)

y_pred = lasso.predict(X_scaled)
mse = mean_squared_error(y, y_pred)
print(f"\nMean Squared Error: {mse:.6f}")

coefficients.to_csv(
    os.path.join(output_dir, "lasso_initial_coefficients.csv"),
    header=["Coefficient"]
)

# -----------------------------
# 8. Cross-validated LASSO
# -----------------------------
cv = RepeatedKFold(n_splits=10, n_repeats=3, random_state=random_state)

lasso_cv = LassoCV(
    alphas=alphas,
    cv=cv,
    random_state=random_state,
    max_iter=10000
)
lasso_cv.fit(X_scaled, y)

mse_path = lasso_cv.mse_path_.mean(axis=1)
mse_std = lasso_cv.mse_path_.std(axis=1)

best_alpha_index = np.argmin(mse_path)
best_alpha = lasso_cv.alphas_[best_alpha_index]

one_se_candidates = np.where(
    mse_path <= mse_path[best_alpha_index] + mse_std[best_alpha_index]
)[0]
one_se_index = one_se_candidates[0]
one_se_alpha = lasso_cv.alphas_[one_se_index]

print(f"\nBest alpha (lambda_min): {best_alpha}")
print(f"1-SE rule alpha (lambda_1se): {one_se_alpha}")

# -----------------------------
# 9. Selected features under lambda_min and lambda_1se
# -----------------------------
lasso_best = Lasso(alpha=best_alpha, max_iter=10000, random_state=random_state)
lasso_best.fit(X_scaled, y)
selected_features_best = X.columns[lasso_best.coef_ != 0].tolist()

lasso_1se = Lasso(alpha=one_se_alpha, max_iter=10000, random_state=random_state)
lasso_1se.fit(X_scaled, y)
selected_features_1se = X.columns[lasso_1se.coef_ != 0].tolist()

print(f"\nSelected features with lambda_min:\n{selected_features_best}")
print(f"\nSelected features with lambda_1se:\n{selected_features_1se}")

pd.DataFrame({"Selected_Features_lambda_min": pd.Series(selected_features_best)}).to_csv(
    os.path.join(output_dir, "lasso_selected_features_lambda_min.csv"),
    index=False
)

pd.DataFrame({"Selected_Features_lambda_1se": pd.Series(selected_features_1se)}).to_csv(
    os.path.join(output_dir, "lasso_selected_features_lambda_1se.csv"),
    index=False
)

# -----------------------------
# 10. Plot MSE vs alpha
# -----------------------------
plt.figure(figsize=(10, 6), dpi=600)
plt.errorbar(
    lasso_cv.alphas_,
    mse_path,
    yerr=mse_std,
    fmt='o',
    ecolor='gray',
    capsize=3
)
plt.axvline(best_alpha, linestyle='--', color='black', label=r'$\lambda_{min}$')
plt.axvline(one_se_alpha, linestyle='--', color='blue', label=r'$\lambda_{1se}$')
plt.xscale('log')
plt.xlabel('Alpha')
plt.ylabel('Mean Squared Error (MSE)')
plt.title('LASSO Regression: MSE vs Alpha')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'lasso_mse_vs_alpha.pdf'), format='pdf')
plt.close()

# -----------------------------
# 11. Plot coefficient paths
# -----------------------------
coefs = []
for a in alphas:
    model = Lasso(alpha=a, max_iter=10000, random_state=random_state)
    model.fit(X_scaled, y)
    coefs.append(model.coef_)

plt.figure(figsize=(10, 6), dpi=600)
ax = plt.gca()
ax.plot(np.log10(alphas), coefs)
plt.xlabel('log10(Alpha)')
plt.ylabel('Coefficients')
plt.title('LASSO Coefficient Paths')
plt.axis('tight')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'lasso_paths.pdf'), format='pdf')
plt.close()

# -----------------------------
# 12. Boruta-LASSO intersection plot
# -----------------------------
# Example Boruta-selected variables in this study
# The final retained variables are defined as the intersection
# used for downstream analysis.
boruta_features = [
    "Age",
    "Gender",
    "Educational",
    "Height",
    "Weight",
    "BMI",
    "Waist_circumference",
    "Hip_circumference",
    "Hypertension",
    "DM",
    "HBV",
    "HCV"
]

lasso_features = [
    "Age",
    "Gender",
    "Educational",
    "Height",
    "Weight",
    "BMI",
    "Waist_circumference",
    "Hip_circumference",
    "Hypertension",
    "DM",
    "Occupation"
]

# Use the reported final retained variables as the intended intersection
intersection = set(final_retained_features)
boruta_set = set(boruta_features)
lasso_set = set(lasso_features)

boruta_only = boruta_set - intersection
lasso_only = lasso_set - intersection

G = nx.Graph()

for feature in boruta_only:
    G.add_edge('Boruta', feature, color='lightcoral')

for feature in lasso_only:
    G.add_edge('LASSO', feature, color='lightblue')

for feature in intersection:
    G.add_edge('Boruta', feature, color='lightcoral')
    G.add_edge('LASSO', feature, color='lightblue')

edge_colors = [data['color'] for _, _, data in G.edges(data=True)]

node_colors = []
for node in G.nodes():
    if node == 'Boruta':
        node_colors.append('lightcoral')
    elif node == 'LASSO':
        node_colors.append('lightblue')
    elif node in boruta_only:
        node_colors.append('lightcoral')
    elif node in lasso_only:
        node_colors.append('lightblue')
    elif node in intersection:
        node_colors.append('plum')

plt.figure(figsize=(18, 10), dpi=600)
pos = nx.spring_layout(G, seed=random_state)

nx.draw_networkx(
    G,
    pos,
    edge_color=edge_colors,
    node_color=node_colors,
    with_labels=True,
    node_size=1800,
    font_size=10,
    edgecolors='none'
)

plt.title('Feature Selection by Boruta and LASSO')
plt.tight_layout()
plt.savefig(
    os.path.join(output_dir, 'feature_selection_boruta_lasso_intersection.pdf'),
    format='pdf'
)
plt.close()

# -----------------------------
# 13. Save final retained variables
# -----------------------------
pd.DataFrame({"Final_Retained_Features": final_retained_features}).to_csv(
    os.path.join(output_dir, "final_retained_features.csv"),
    index=False
)

# -----------------------------
# 14. Save package/version information
# -----------------------------
import sys
import sklearn
import matplotlib
import networkx

version_info = pd.DataFrame({
    "Package": ["python", "pandas", "numpy", "matplotlib", "scikit-learn", "networkx"],
    "Version": [
        sys.version.replace("\n", " "),
        pd.__version__,
        np.__version__,
        matplotlib.__version__,
        sklearn.__version__,
        networkx.__version__
    ]
})

version_info.to_csv(
    os.path.join(output_dir, "session_versions_lasso_python.csv"),
    index=False
)

print("\nLASSO feature selection and Boruta-LASSO intersection plotting completed successfully.")
print(f"Results saved to: {output_dir}")